# 02 - Delta Lakehouse: Bronze / Silver / Gold (Rubric item 2 / 25 pts)

**Project:** ShopSense | **Program:** SDAIA Academy - Modern Data Engineering for AI Systems

## What this notebook must prove
| Rubric requirement | Where it is proven |
|---|---|
| **Delta Lake** used for real (not pandas/Parquet) | Section 2 - `pyspark` + `delta-spark`, every table written with `format('delta')` |
| **Bronze / Silver / Gold** layers | Sections 4, 5, 7 |
| A **real MERGE (upsert)** keyed on a business key | Section 5.3 - `MERGE` on `order_id`, run twice, second run both updates and inserts |
| **Schema enforcement demonstrated** | Section 6 - a bad write is *refused* by Delta, with the error captured |
| Gold is a **genuine aggregate**, not a copy of Silver | Section 7 - daily revenue by category/city + customer segments |

Input: the Bronze landing file produced by notebook `01_ingestion_kafka.ipynb`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import os, json, shutil, datetime

PROJECT   = Path('/content/drive/MyDrive/sdaia_capstone')
LANDING   = PROJECT / 'data' / 'bronze_landing'
REPORTS   = PROJECT / 'reports'
LOCAL_LAKE = Path('/content/lakehouse')          # Delta tables are built on local disk (fast),
DRIVE_LAKE = PROJECT / 'lakehouse'               # then copied to Drive at the end (durable).

LOCAL_LAKE.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)

landing_files = sorted(LANDING.glob('orders_valid_*.jsonl'))
assert landing_files, f'No landing file found in {LANDING}. Run notebook 01 first.'
LANDING_FILE = landing_files[-1]
print('using landing file :', LANDING_FILE)
print('records            :', sum(1 for _ in open(LANDING_FILE)))

## 2. Install Spark + Delta Lake

`delta-spark 3.3.0` is the matching Delta release for Spark 3.5.x. `configure_spark_with_delta_pip`
pulls the Delta JARs from Maven the first time the session starts - give it a minute.

In [ ]:
!pip install -q pyspark==3.5.3 delta-spark==3.3.0

In [ ]:
import subprocess, os
if subprocess.run(['bash','-lc','java -version'], capture_output=True).returncode != 0:
    !apt-get -qq install -y openjdk-17-jdk-headless
print(subprocess.run(['bash','-lc','java -version'], capture_output=True, text=True).stderr.strip())

In [ ]:
from pyspark.sql import SparkSession
from delta import configure_spark_with_delta_pip
import delta, pyspark

builder = (
    SparkSession.builder
    .appName('shopsense-lakehouse')
    .master('local[*]')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.databricks.delta.schema.autoMerge.enabled', 'false')  # keep enforcement strict
    .config('spark.sql.warehouse.dir', '/content/spark-warehouse')
)
spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

print('Spark   :', spark.version)
print('Delta   :', delta.__name__, pyspark.__version__)
print('Delta extension active:', spark.conf.get('spark.sql.extensions'))

## 3. Table paths

| Layer | Path | Contract |
|---|---|---|
| Bronze | `lakehouse/bronze/orders` | append-only, as ingested, plus audit columns |
| Silver | `lakehouse/silver/orders` | one row per `order_id` (business key), typed and cleaned, kept current by `MERGE` |
| Gold | `lakehouse/gold/daily_category_revenue` | daily aggregate per category and city |
| Gold | `lakehouse/gold/customer_segments` | per-customer aggregate + segment label |

In [ ]:
BRONZE = str(LOCAL_LAKE / 'bronze' / 'orders')
SILVER = str(LOCAL_LAKE / 'silver' / 'orders')
GOLD_REVENUE  = str(LOCAL_LAKE / 'gold' / 'daily_category_revenue')
GOLD_CUSTOMER = str(LOCAL_LAKE / 'gold' / 'customer_segments')
for p in (BRONZE, SILVER, GOLD_REVENUE, GOLD_CUSTOMER):
    print(p)

## 4. Bronze - land the validated stream as-is

Bronze keeps the data exactly as it arrived (plus ingestion audit columns).
No business logic here, so we can always replay Silver and Gold from it.

In [ ]:
from pyspark.sql import functions as F, types as T

bronze_schema = T.StructType([
    T.StructField('order_id',        T.StringType()),
    T.StructField('customer_id',     T.StringType()),
    T.StructField('customer_email',  T.StringType()),
    T.StructField('product_id',      T.StringType()),
    T.StructField('product_name',    T.StringType()),
    T.StructField('category',        T.StringType()),
    T.StructField('quantity',        T.IntegerType()),
    T.StructField('unit_price',      T.DoubleType()),
    T.StructField('currency',        T.StringType()),
    T.StructField('city',            T.StringType()),
    T.StructField('status',          T.StringType()),
    T.StructField('order_ts',        T.StringType()),
    T.StructField('delivered_ts',    T.StringType()),
    T.StructField('kafka_topic',     T.StringType()),
    T.StructField('kafka_partition', T.IntegerType()),
    T.StructField('kafka_offset',    T.LongType()),
    T.StructField('ingested_at',     T.StringType()),
])

raw_df = (spark.read.schema(bronze_schema).json(str(LANDING_FILE))
          .withColumn('_bronze_loaded_at', F.current_timestamp())
          .withColumn('_source_file', F.lit(LANDING_FILE.name)))

(raw_df.write.format('delta').mode('overwrite').save(BRONZE))

bronze_df = spark.read.format('delta').load(BRONZE)
print('bronze rows :', bronze_df.count())
print('distinct order_id:', bronze_df.select('order_id').distinct().count(), '(duplicates are expected - Silver resolves them)')
bronze_df.show(5, truncate=False)

In [ ]:
# Proof this really is a Delta table and not just Parquet
!ls -R /content/lakehouse/bronze/orders/_delta_log | head -20

## 5. Silver - typed, cleaned, one row per business key, maintained by MERGE

### 5.1 Transform Bronze -> Silver shape

In [ ]:
silver_ready = (
    bronze_df
    .withColumn('order_ts',     F.to_timestamp('order_ts'))
    .withColumn('delivered_ts', F.to_timestamp('delivered_ts'))
    .withColumn('ingested_at',  F.to_timestamp('ingested_at'))
    .withColumn('order_date',   F.to_date('order_ts'))
    .withColumn('line_total',   F.round(F.col('quantity') * F.col('unit_price'), 2))
    .withColumn('is_revenue',   F.col('status').isin('paid', 'shipped', 'delivered'))
    .withColumn('city',         F.initcap(F.trim('city')))
    .withColumn('category',     F.lower(F.trim('category')))
    .select('order_id','customer_id','customer_email','product_id','product_name','category',
            'quantity','unit_price','line_total','currency','city','status','is_revenue',
            'order_ts','delivered_ts','order_date','ingested_at')
)
silver_ready.printSchema()

### 5.2 Deduplicate the source before merging

`MERGE` refuses a source that contains the same key twice - which is correct, because the
engine cannot know which version wins. We resolve it explicitly: **latest `ingested_at` per `order_id`**.

In [ ]:
from pyspark.sql.window import Window

w = Window.partitionBy('order_id').orderBy(F.col('ingested_at').desc(), F.col('order_ts').desc())

def dedupe(df):
    return (df.withColumn('_rn', F.row_number().over(w))
              .filter(F.col('_rn') == 1)
              .drop('_rn'))

print('before dedupe :', silver_ready.count())
print('after  dedupe :', dedupe(silver_ready).count())

### 5.3 The MERGE (upsert) on the business key `order_id`

We deliberately split the data into two batches to prove the upsert really works:

* **Batch 1** - the first 70% of orders -> everything is an INSERT.
* **Batch 2** - the remaining 30% **plus** the orders whose `status` changed to `delivered`
  -> a mix of INSERTs and UPDATEs on the same key.

In [ ]:
from delta.tables import DeltaTable

all_orders = dedupe(silver_ready).cache()
ids = [r.order_id for r in all_orders.select('order_id').orderBy('order_id').collect()]
cut = int(len(ids) * 0.7)
batch1_ids, batch2_ids = set(ids[:cut]), set(ids[cut:])

# batch 1: the original (pre-update) version of the first 70% of orders
batch1 = (silver_ready.filter(F.col('order_id').isin(list(batch1_ids)))
                      .withColumn('_rn', F.row_number().over(
                          Window.partitionBy('order_id').orderBy(F.col('ingested_at').asc())))
                      .filter(F.col('_rn') == 1).drop('_rn'))

# batch 2: the last 30% (new keys) + the latest version of the first 70% (existing keys, changed status)
batch2 = all_orders.filter(F.col('order_id').isin(list(batch2_ids)) |
                           F.col('order_id').isin(list(batch1_ids)))

print('batch 1 rows:', batch1.count())
print('batch 2 rows:', batch2.count())

In [ ]:
# --- Batch 1: create the Silver table -------------------------------------------------
batch1.write.format('delta').mode('overwrite').save(SILVER)
silver_tbl = DeltaTable.forPath(spark, SILVER)
print('silver rows after batch 1 :', spark.read.format('delta').load(SILVER).count())

In [ ]:
# --- Batch 2: the actual MERGE / upsert ------------------------------------------------
(silver_tbl.alias('t')
   .merge(batch2.alias('s'), 't.order_id = s.order_id')          # <-- business key
   .whenMatchedUpdate(
        condition = 's.ingested_at > t.ingested_at OR s.status <> t.status',
        set = {c: F.col(f's.{c}') for c in batch2.columns})
   .whenNotMatchedInsertAll()
   .execute())

silver_df = spark.read.format('delta').load(SILVER)
print('silver rows after MERGE :', silver_df.count())
print('distinct order_id       :', silver_df.select('order_id').distinct().count(),
      ' <- must be equal: one row per business key')

In [ ]:
# Delta records exactly what the MERGE did - this is the evidence the upsert was real
hist = (spark.sql(f"DESCRIBE HISTORY delta.`{SILVER}`")
             .select('version', 'operation', 'operationMetrics'))
hist.show(truncate=False)

metrics = hist.orderBy(F.col('version').desc()).first()['operationMetrics']
print('\nMERGE metrics:')
for k in ['numTargetRowsInserted', 'numTargetRowsUpdated', 'numTargetRowsMatchedUpdated',
          'numSourceRows', 'numOutputRows']:
    if k in metrics:
        print(f'  {k:32} {metrics[k]}')

In [ ]:
# Same thing expressed as SQL, for the report (Delta MERGE INTO is standard SQL here)
print(f'''
MERGE INTO delta.`{SILVER}` AS t
USING updates AS s
   ON t.order_id = s.order_id
 WHEN MATCHED AND s.ingested_at > t.ingested_at THEN UPDATE SET *
 WHEN NOT MATCHED THEN INSERT *
''')

## 6. Schema enforcement - prove a bad write is refused

The rubric asks for a bad write **actually being refused**, not just described.
Two attempts below, both must fail:

1. an **extra undeclared column** (`discount_hack`),
2. a **type violation** (`quantity` as a string).

Delta's schema enforcement rejects both, so no corrupt data ever lands in Silver.

In [ ]:
from pyspark.sql.utils import AnalysisException

enforcement_log = []

def attempt(name, df, path=SILVER):
    global enforcement_log
    try:
        df.write.format('delta').mode('append').save(path)
        enforcement_log.append((name, 'ACCEPTED  <-- unexpected!'))
        print(f'[{name}] write ACCEPTED - schema enforcement did NOT trigger')
    except Exception as exc:
        head = str(exc).strip().split('\n')[0][:220]
        enforcement_log.append((name, f'REFUSED: {head}'))
        print(f'[{name}] write REFUSED by Delta')
        print('   ', head, '\n')

# 1) undeclared extra column
bad_extra = silver_df.limit(2).withColumn('discount_hack', F.lit(True))
attempt('extra_undeclared_column', bad_extra)

# 2) wrong type on an existing column
bad_type = (silver_df.limit(2)
            .withColumn('quantity', F.lit('three').cast(T.StringType())))
attempt('wrong_type_quantity', bad_type)

In [ ]:
print('silver rows after the two bad writes :', spark.read.format('delta').load(SILVER).count(),
      ' <- unchanged, nothing corrupt landed')
for name, outcome in enforcement_log:
    print(f'  {name:28} -> {outcome[:110]}')

In [ ]:
# The controlled way to evolve a schema - opt in explicitly, per write
(silver_df.limit(0).withColumn('promo_code', F.lit(None).cast(T.StringType()))
   .write.format('delta').mode('append').option('mergeSchema', 'true').save(SILVER))
print('columns after an explicit, opted-in schema evolution:')
print(spark.read.format('delta').load(SILVER).columns)

## 7. Gold - genuine aggregates

Gold is **not** a copy of Silver. Each Gold table answers a business question and has a
different grain from Silver (which is one row per order).

In [ ]:
silver_df = spark.read.format('delta').load(SILVER)

gold_revenue = (
    silver_df
      .filter(F.col('is_revenue'))                          # cancelled/created orders are not revenue
      .groupBy('order_date', 'category', 'city')
      .agg(
          F.countDistinct('order_id').alias('orders_count'),
          F.countDistinct('customer_id').alias('unique_customers'),
          F.sum('quantity').alias('units_sold'),
          F.round(F.sum('line_total'), 2).alias('total_revenue_sar'),
          F.round(F.avg('line_total'), 2).alias('avg_order_value_sar'),
          F.round(F.max('line_total'), 2).alias('largest_order_sar'),
      )
      .withColumn('revenue_per_customer',
                  F.round(F.col('total_revenue_sar') / F.col('unique_customers'), 2))
      .orderBy(F.col('order_date').desc(), F.col('total_revenue_sar').desc())
)

gold_revenue.write.format('delta').mode('overwrite').partitionBy('order_date').save(GOLD_REVENUE)

print(f'silver grain : one row per order      -> {silver_df.count()} rows')
print(f'gold  grain  : date x category x city -> {spark.read.format("delta").load(GOLD_REVENUE).count()} rows')
spark.read.format('delta').load(GOLD_REVENUE).show(12, truncate=False)

In [ ]:
gold_customer = (
    silver_df
      .groupBy('customer_id')
      .agg(
          F.countDistinct('order_id').alias('lifetime_orders'),
          F.round(F.sum(F.when(F.col('is_revenue'), F.col('line_total')).otherwise(0.0)), 2)
           .alias('lifetime_value_sar'),
          F.round(F.avg('line_total'), 2).alias('avg_basket_sar'),
          F.min('order_date').alias('first_order_date'),
          F.max('order_date').alias('last_order_date'),
          F.countDistinct('category').alias('categories_bought'),
          F.sum(F.when(F.col('status') == 'cancelled', 1).otherwise(0)).alias('cancelled_orders'),
      )
      .withColumn('cancellation_rate',
                  F.round(F.col('cancelled_orders') / F.col('lifetime_orders'), 3))
      .withColumn('segment',
          F.when(F.col('lifetime_value_sar') >= 3000, 'vip')
           .when(F.col('lifetime_value_sar') >= 1000, 'loyal')
           .when(F.col('lifetime_orders') >= 2, 'returning')
           .otherwise('new'))
)

gold_customer.write.format('delta').mode('overwrite').save(GOLD_CUSTOMER)
spark.read.format('delta').load(GOLD_CUSTOMER).orderBy(F.col('lifetime_value_sar').desc()).show(10)
spark.read.format('delta').load(GOLD_CUSTOMER).groupBy('segment').count().orderBy('count', ascending=False).show()

### 7.1 A business question answered straight off Gold

In [ ]:
spark.read.format('delta').load(GOLD_REVENUE).createOrReplaceTempView('gold_revenue')
spark.sql('''
    SELECT category,
           ROUND(SUM(total_revenue_sar), 2) AS revenue_sar,
           SUM(orders_count)                AS orders,
           ROUND(SUM(total_revenue_sar) / SUM(orders_count), 2) AS aov_sar
      FROM gold_revenue
     GROUP BY category
     ORDER BY revenue_sar DESC
''').show()

## 8. Time travel - read Silver as it was before the MERGE

In [ ]:
v0 = spark.read.format('delta').option('versionAsOf', 0).load(SILVER)
now = spark.read.format('delta').load(SILVER)
print('silver @ version 0 :', v0.count(), 'rows')
print('silver @ latest    :', now.count(), 'rows')

delivered_v0  = v0.filter(F.col('status') == 'delivered').count()
delivered_now = now.filter(F.col('status') == 'delivered').count()
print(f"orders in status 'delivered'  before MERGE: {delivered_v0}  ->  after MERGE: {delivered_now}")

## 9. Persist the lakehouse to Drive + write the stage report

In [ ]:
import shutil
if DRIVE_LAKE.exists():
    shutil.rmtree(DRIVE_LAKE)
shutil.copytree(LOCAL_LAKE, DRIVE_LAKE)
print('lakehouse copied to', DRIVE_LAKE)
!du -sh /content/drive/MyDrive/sdaia_capstone/lakehouse

In [ ]:
run_id = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%SZ')
report = {
    'run_id': run_id,
    'stage': 'lakehouse',
    'engine': f'pyspark {pyspark.__version__} + delta-spark 3.3.0',
    'source_file': LANDING_FILE.name,
    'bronze_rows': bronze_df.count(),
    'silver_rows': silver_df.count(),
    'silver_distinct_keys': silver_df.select('order_id').distinct().count(),
    'merge_metrics': {k: v for k, v in metrics.items()},
    'schema_enforcement': dict(enforcement_log),
    'gold_daily_category_revenue_rows': spark.read.format('delta').load(GOLD_REVENUE).count(),
    'gold_customer_segments_rows': spark.read.format('delta').load(GOLD_CUSTOMER).count(),
    'paths': {'bronze': BRONZE, 'silver': SILVER,
              'gold_revenue': GOLD_REVENUE, 'gold_customer': GOLD_CUSTOMER},
    'finished_at': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
out = REPORTS / f'lakehouse_report_{run_id}.json'
out.write_text(json.dumps(report, indent=2, default=str))
print(json.dumps(report, indent=2, default=str))

In [ ]:
spark.stop()
print('Spark session stopped.')

## Appendix - Plan B if Spark will not start in Colab

Everything above can also be done with `deltalake` (delta-rs), which the rubric accepts.
It needs no Java and installs in seconds. Use this **only** if the Spark cell above fails.

```python
!pip install -q deltalake pandas pyarrow
import pandas as pd
from deltalake import DeltaTable, write_deltalake

df = pd.read_json(LANDING_FILE, lines=True)

# Bronze
write_deltalake('/content/lakehouse_rs/bronze/orders', df, mode='overwrite')

# Silver: dedupe on the business key, then a real MERGE
silver = df.sort_values('ingested_at').drop_duplicates('order_id', keep='last')
write_deltalake('/content/lakehouse_rs/silver/orders', silver.head(200), mode='overwrite')

dt = DeltaTable('/content/lakehouse_rs/silver/orders')
(dt.merge(source=silver, predicate='target.order_id = source.order_id',
          source_alias='source', target_alias='target')
   .when_matched_update_all()
   .when_not_matched_insert_all()
   .execute())

# Schema enforcement: this raises, which is the point
bad = silver.head(2).assign(discount_hack=True)
try:
    write_deltalake('/content/lakehouse_rs/silver/orders', bad, mode='append')
except Exception as exc:
    print('REFUSED by Delta:', exc)

# Gold: a genuine aggregate
silver['order_date'] = pd.to_datetime(silver['order_ts']).dt.date
silver['line_total'] = silver['quantity'] * silver['unit_price']
gold = (silver[silver.status.isin(['paid','shipped','delivered'])]
        .groupby(['order_date','category','city'], as_index=False)
        .agg(orders_count=('order_id','nunique'),
             total_revenue_sar=('line_total','sum'),
             avg_order_value_sar=('line_total','mean')))
write_deltalake('/content/lakehouse_rs/gold/daily_category_revenue', gold, mode='overwrite')
```